In [44]:
my_list = [1,3,5,4,1,9,8,7,6]

In [45]:
def the_oracle(my_input):
    winner = 7
    if my_input == winner:
        return True
    else:
        return False

In [46]:
for index, trial_number in enumerate(my_list):
    if the_oracle(trial_number) is True:
        print('Winner found at index %i' % index)
        print('%i calls to the Oracle used' % (index + 1))
        break

Winner found at index 7
8 calls to the Oracle used


In [47]:
# on an average we need N/2 calls

In [48]:
# so oracle can decide what to states to flip ( we define our winner in this balckbox)
# we will encode an unitary matrix to flip these states and place it in a quantum circuit
# we will need a quantum circuit with gates and conveniently controlled-Z gate does this (flip 11 to -11)
# also we require reflection operator.
# this is the base of grovers diffusion operator = oracle + reflection

In [49]:
from qiskit import *
from qiskit_aer import Aer
import numpy as np
import matplotlib.pyplot as plt

In [50]:
#define circuit
oracle = QuantumCircuit(2,name="oracle") # 2 qubits
oracle.cz(0,1) #control z flips the winner
oracle_gate = oracle.to_gate()  # Convert to gate and assign to variable
oracle.draw()


q_0: ─■─
      │ 
q_1: ─■─

In [51]:
# Choose backend (statevector simulator lets us see the quantum state)
backend = Aer.get_backend('statevector_simulator')
# Create a Grover circuit with 2 qubits and 2 classical bits
grover_circ = QuantumCircuit(2, 2)
# Apply Hadamard gates to both qubits (superposition)
grover_circ.h([0, 1])
# Apply oracle directly (CZ gate flips the phase of |11> state)
grover_circ.cz(0, 1)
# Draw the circuit
grover_circ.draw()

┌───┐   
q_0: ┤ H ├─■─
     ├───┤ │ 
q_1: ┤ H ├─■─
     └───┘   
c: 2/════════

In [52]:
# Execute the circuit on the simulator (modern Qiskit 2.x approach)
job = backend.run(grover_circ)
result = job.result()

# Get statevector of the system
sv = result.get_statevector()
# Round to 2 decimal places for readability
print(np.around(sv, 2))

[ 0.5+0.j  0.5+0.j  0.5+0.j -0.5+0.j]


In [53]:
# The circuit starts with 2 qubits in state ∣00⟩
# The Hadamard gates put both qubits into an equal superposition
# The oracle flips the phase of the "marked state"

In [54]:
#finally we need to use reflection operator to amplify the amplitude of the marked state
reflection = QuantumCircuit(2,name="reflection")
reflection.h([0,1])
reflection.z([0,1])
reflection.cz(0,1)
reflection.h([0,1])
reflection_gate = reflection.to_gate()  # Convert to gate and assign to variable
reflection.draw()



┌───┐┌───┐   ┌───┐
q_0: ┤ H ├┤ Z ├─■─┤ H ├
     ├───┤├───┤ │ ├───┤
q_1: ┤ H ├┤ Z ├─■─┤ H ├
     └───┘└───┘   └───┘

In [57]:
backend = Aer.get_backend('qasm_simulator')
# Create Grover circuit with 2 qubits and 2 classical bits
grover_circ = QuantumCircuit(2, 2)

# Step 1: Put qubits into superposition
grover_circ.h([0, 1])

# Step 2: Apply the oracle (CZ gate marks the |11> state)
grover_circ.cz(0, 1)

# Step 3: Apply the reflection (diffusion operator)
# Reflection = H + Z + CZ + H
grover_circ.h([0, 1])  # H gates
grover_circ.z([0, 1])  # Z gates  
grover_circ.cz(0, 1)   # CZ gate
grover_circ.h([0, 1])  # H gates

# Step 4: Measure both qubits
grover_circ.measure([0, 1], [0, 1])

grover_circ.draw()

┌───┐   ┌───┐┌───┐   ┌───┐┌─┐   
q_0: ┤ H ├─■─┤ H ├┤ Z ├─■─┤ H ├┤M├───
     ├───┤ │ ├───┤├───┤ │ ├───┤└╥┘┌─┐
q_1: ┤ H ├─■─┤ H ├┤ Z ├─■─┤ H ├─╫─┤M├
     └───┘   └───┘└───┘   └───┘ ║ └╥┘
c: 2/═══════════════════════════╩══╩═
                                0  1

In [61]:
# Run the circuit with more shots to see the probability distribution
job = backend.run(grover_circ, shots=1024)
result = job.result()
counts = result.get_counts(grover_circ)
print("Measurement counts:", counts)





Measurement counts: {'11': 1024}
